## Benchmark with other methods

Categories:
- (Multi-omics) latent representation
- Gradient computation & Trajectory inference

In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt

from torch_geometric import utils as pyg_utils
from scipy.stats import spearmanr
from sklearn.metrics import normalized_mutual_info_score

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, trajectory, metrics
import vgae
import scFates as scf

In [ ]:
%load_ext autoreload
%autoreload 2

Load sample data: <br>

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
ab_path = '../data/integrated/antibody/'

sample_id = 'NIH_F5'

adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

# Antibody (validation)
adata_ab = IO.load_ab_stain(
    os.path.join(ab_path, sample_id+'.ome.tif'),
    adata_ref=adata_xenium
)

# Normalize to [0, 1] per channel
scaled_chans = np.zeros_like(adata_ab.X)
for i, chan in enumerate(adata_ab.X.T):
    chan = (chan-chan.min()) / (chan.max()-chan.min())
    scaled_chans[:, i] = chan
adata_ab.X = scaled_chans

ab_dict = {
    'Opal 690-GS': 'Central Vein',
    'Opal 780-CYP3A4': 'Peri-central',
    'Opal 570-ASS1': 'Peri-portal',
    'Opal 520-Col1': 'Portal Vein'
}

ab_labels = list(ab_dict.keys())

### Create synthetic ground-truth trajectory from antibody

In [ ]:
from scipy.spatial import cKDTree

def correct_mislabel_veins(adata, use_rep='ab_label', k=10):
    spatial_coords = adata.obsm["spatial"]
    labels = adata.obs[use_rep].values

    # Get indices for each label
    idx_label_1 = np.where(labels == 1)[0]
    idx_label_2 = np.where(labels == 2)[0]
    idx_label_3 = np.where(labels == 3)[0]

    # Build KD-trees for fast nearest-neighbor search
    tree_label_1 = cKDTree(spatial_coords[idx_label_1])
    tree_label_2 = cKDTree(spatial_coords[idx_label_2])

    # Find average distances from each label 3 cell to label 1 and label 2
    d1, _ = tree_label_1.query(spatial_coords[idx_label_3], k=k, workers=-1)
    d2, _ = tree_label_2.query(spatial_coords[idx_label_3], k=k, workers=-1)

    avg_d1 = d1.mean(axis=1)
    avg_d2 = d2.mean(axis=1)

    # Identify mislabeled 3s where avg distance to 1 is smaller than to 2
    mislabeled = avg_d1 < avg_d2
    labels[idx_label_3[mislabeled]] = 0  # Correct misclassified labels

    # Denoise
    labels = adata.obs[use_rep].values

    # Get indices for labels 0 & 3
    idx_label_0 = np.where(labels == 0)[0]
    idx_label_3 = np.where(labels == 3)[0]
    idx_0_3 = np.concatenate([idx_label_0, idx_label_3])  # Only process 0 & 3

    # Build KD-tree for spatial queries
    tree = cKDTree(spatial_coords)

    # Query nearest neighbors (excluding self)
    _, neighbors = tree.query(spatial_coords[idx_0_3], k=k+1, workers=-1)  # k+1 to exclude self

    # Count majority labels in neighbors
    for i, idx in enumerate(idx_0_3):
        neighbor_labels = labels[neighbors[i, 1:]]  # Exclude self
        majority_label = np.bincount(neighbor_labels).argmax()  # Most frequent label

        # Only update if the majority is different from the current label
        if majority_label in {0, 3} and majority_label != labels[idx]:
            labels[idx] = majority_label

    # Update AnnData object
    adata.obs[use_rep] = labels
    adata.obs[use_rep] = adata.obs[use_rep].astype('category')
    return None


def calculate_vein_axis(
    adata, use_rep='ab_label', 
    w1=.5, w2=.5, 
    vmin=0., vmax=1., k=10
):
    r"""Approx. annotation of PV -> CV axis from only antibody imaging"""
    # Extract spatial coordinates
    coords = adata.obsm['spatial']

    # Identify indices for each structure
    cv_indices = np.where(adata.obs[use_rep] == 0)[0]
    pp_indices = np.where(adata.obs[use_rep] == 2)[0]
    pv_indices = np.where(adata.obs[use_rep] == 3)[0]

    # Build KD-Trees for each structure
    cv_tree = cKDTree(coords[cv_indices])
    pp_tree = cKDTree(coords[pp_indices])
    pv_tree = cKDTree(coords[pv_indices])

    # Initialize array to hold PV-CV values
    axis_values = np.zeros(coords.shape[0])

    # Calculate mean distances to each structure
    for i, point in enumerate(coords):
        d_cv, _ = cv_tree.query(point, k=k)
        d_pp, _ = pp_tree.query(point, k=k)
        d_pv, _ = pv_tree.query(point, k=k)

        mu_cv = np.mean(d_cv)
        mu_pp = np.mean(d_pp)
        mu_pv = np.mean(d_pv)

        # Calculate H1(Medulla vs. Cortex) & H2 (Cortex vs. Capsule)
        H1 = (mu_pp - mu_pv) / (mu_pp + mu_pv)
        H2 = (mu_cv - mu_pp) / (mu_cv + mu_pp)

        # Combine H1 and H2 to get CMA value
        axis_values[i] = w1 * H1 + w2 * H2

    # Rescale to [vmin, vmax]
    axis_min, axis_max = np.min(axis_values), np.max(axis_values)
    axis_values = vmin + (axis_values - axis_min) * ((vmax-vmin) / (axis_max-axis_min))

    adata.obs['t'] = 1.0 - axis_values   # PV (0) --> CV (1)
    return None



In [ ]:
# Obtain 1-hot encoded argmax
argmax_expr = adata_ab.X.argmax(1)
adata_ab.obs['ab_label'] = argmax_expr
correct_mislabel_veins(adata_ab, k=50)
calculate_vein_axis(adata_ab, k=10)


In [ ]:
sq.pl.spatial_scatter(
    adata_ab, color='ab_label', groups=[0, 3], img=False, size=30
)

sq.pl.spatial_scatter(
    adata_ab, color='t', img=False, size=30, cmap='RdBu_r'
)

# Save "silver-standard" annotation
np.save('../results/liver/antibody_gamma.npy', adata_ab.obs['t'].values)

### PCA, Diffusion map, DPT

In [ ]:
sc.pp.normalize_total(adata_xenium)
sc.pp.log1p(adata_xenium)
sc.pp.pca(adata_xenium)
sc.pp.neighbors(adata_xenium)
sc.tl.diffmap(adata_xenium)

Use default PCA (`50`) & Diffmap (`15`) dimensions, rotate with `DPT` gene expression direction

In [ ]:
curve = trajectory.get_curve(adata_xenium, use_rep='X_pca')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')
adata_xenium.obs['t_pca'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

In [ ]:
curve = trajectory.get_curve(adata_xenium, use_rep='X_diffmap')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')

adata_xenium.obs['t_diffmap'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

#  sq.pl.spatial_scatter(adata_xenium, color=['t_pca', 't_diffmap'], img=False, size=20, cmap='RdBu_r')


In [ ]:
adata_xenium.uns['iroot'] = adata_xenium[:, 'DPT'].X.A.argmax()
sc.tl.dpt(adata_xenium)

t_dpt = adata_xenium.obs['dpt_pseudotime'].copy()
t_dpt = (t_dpt-t_dpt.min()) / (t_dpt.max()-t_dpt.min())
adata_xenium.obs['t_dpt'] = t_dpt
adata_xenium.obs.drop('dpt_pseudotime', inplace=True, axis=1)
# sq.pl.spatial_scatter(adata_xenium, color='t_dpt', img=False, size=20, cmap='RdBu_r')

### SIMVI

In [ ]:
adata_xenium.obsm['X_simvi'] = np.load('../results/liver/SIMVI_xenium_s20.npy')
curve = trajectory.get_curve(adata_xenium, use_rep='X_simvi')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')

adata_xenium.obs['t_simvi'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

### SpaceFlow

In [ ]:
t_spaceflow = pd.read_csv('../results/liver/SpaceFlow_50_pseudotime.csv', index_col=[0])['pSM_spaceflow'].values
adata_xenium.obs['t_spaceflow'] = t_spaceflow

### CORAL

In [ ]:
t_coral = pd.read_csv('../results/liver/Coral_pseudotime.csv', index_col=[0])['t'].values
adata_xenium.obs['t_coral'] = t_coral

# sq.pl.spatial_scatter(
#     adata_xenium, color='t_coral', 
#     cmap='RdBu_r', size=20, img=False,
#     title=r'Spatial Trajectory ($\gamma(t)$)'+'\nCORAL (Xenium)'
# )


### GASTON

In [ ]:
adata_xenium_gaston = sc.read_h5ad('../results/liver/adata_xenium_gaston_dim10.h5')
adata_xenium_gaston = adata_xenium_gaston[adata_xenium.obs_names]
adata_xenium.obs['t_gaston'] = adata_xenium_gaston.obs['gaston_isodepth'].values.copy()

# sq.pl.spatial_scatter(
#     adata_xenium_gaston, color='isodepth', 
#     cmap='RdBu_r', size=20, img=False,
#     title=r'Spatial Trajectory ($\gamma(t)$)'+'\nGASTON(Xenium)',
#     vmin=0.25, vmax=0.65
# )

del adata_xenium_gaston
gc.collect()


### scVI / TotalVI

In [ ]:
import scvi
scvi.settings.seed = 42
print("Last run with scvi-tools version:", scvi.__version__)

#### scVI

Model training:

In [ ]:
scvi.model.SCVI.setup_anndata(adata_xenium)

model = scvi.model.SCVI(
    adata_xenium, 
    n_layers=2, 
    gene_likelihood="nb"
)

model.train()

In [ ]:
latent = model.get_latent_representation()
np.save('../results/liver/scvi_10.npy', latent)
adata_xenium.obsm['X_scvi'] = latent

Trajectory inference:

In [ ]:
n_latent = 10
adata_xenium.obsm['X_scvi'] = np.load('../results/liver/scvi_{}.npy'.format(n_latent))
curve = trajectory.get_curve(adata_xenium, use_rep='X_scvi')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')
adata_xenium.obs['t_scvi'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

# Spatial visualization
# sq.pl.spatial_scatter(
#     adata_xenium, color='t_scvi', 
#     cmap='RdBu_r', size=20, img=False,
#     title=r'Spatial gradient $\gamma(t)$ - scVI',
# )

Antibody validation:

Spatial heterogeneity metrics:

- (1). Moran's I, Geary's C (Ripley: measure cell density instead)
- (2). Network's centrality
  - degree centrality: % non-group members connected to group members
  - closeness centrality: closeness the group to other nodes

In [ ]:
# Spatial heterogeneity metrics
sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_scvi = adata.uns['moranI'].loc['t', 'I']
geary_scvi = adata.uns['gearyC'].loc['t', 'C']
centrality_scvi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


#### TotalVI

In [ ]:
def impute_lowres_modality(adata_desi, adata_xenium):
    """Impute low-res `adata_desi` to high-res `adata_xenium"""
    coord_map = {tuple(adata_desi.obsm['spatial'][i]): i for i in range(adata_desi.shape[0])}
    expr_imputed = np.zeros((adata_xenium.shape[0], adata_desi.shape[1]))

    for i in range(adata_xenium.shape[0]):
        expr_imputed[i] = adata_desi[coord_map[tuple(adata_xenium.obsm['desi_map'][i])]].X.toarray()
    adata_imputed = sc.AnnData(expr_imputed)
    adata_imputed.obs_names = adata_xenium.obs_names
    adata_imputed.var_names = adata_desi.var_names
    
    return adata_imputed

In [ ]:
import muon
import mudata as md

# Impute cell-level metabolite expressions
adata_desi_imputed = impute_lowres_modality(adata_desi, adata_xenium)

# Create joint-modality data, note: totalVI requires "unnormlized" 2nd-modality intensities
adata_desi_imputed.X = (255*adata_desi_imputed.X).astype(np.uint8)
mdata = md.MuData({'rna': adata_xenium, 'metabolite': adata_desi_imputed})
mdata

In [ ]:
scvi.model.TOTALVI.setup_mudata(
    mdata,
    rna_layer=None,
    protein_layer=None,
    modalities={
        "rna_layer": "rna",
        "protein_layer": "metabolite",
    },
)

model = scvi.model.TOTALVI(mdata)  # n_latent defaults to 20...
model.train()


In [ ]:
# Obtain joint latent representation
adata_xenium.obsm['X_totalvi'] = model.get_latent_representation()
np.save('../results/liver/totalvi_{}.npy'.format(adata_xenium.obsm['X_totalvi'].shape[1]), adata_xenium.obsm['X_totalvi'])

Trajectory inference:

In [ ]:
n_latent = 20
latent = np.load('../results/liver/totalvi_{}.npy'.format(n_latent))
adata_xenium.obsm['X_totalvi'] = latent

curve = trajectory.get_curve(adata_xenium, use_rep='X_totalvi')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')

adata_xenium.obs['t_totalvi'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

# Spatial visualization
# sq.pl.spatial_scatter(
#     adata_xenium, color='t_totalvi', 
#     cmap='RdBu_r', size=20, img=False,
#     title=r'Spatial grdient $\gamma(t)$ - TotalVI',
# )

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata_xenium, cluster_key='milestones')

display(adata_xenium.uns['moranI'])
display(adata_xenium.uns['gearyC'])
display(adata_xenium.uns['milestones_centrality_scores'])

moran_totalvi = adata.uns['moranI'].loc['t', 'I']
geary_totalvi = adata.uns['gearyC'].loc['t', 'C']
centrality_totalvi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### CLUE

In [ ]:
import anndata as ad
import scglue
import networkx as nx

In [ ]:
# Configure model
scglue.models.configure_dataset(
    adata, "NB", use_highly_variable=False
)

scglue.models.configure_dataset(
    adata_desi, "Normal", use_highly_variable=False
)

In [ ]:
# First try without pretraining
n_latent = 8
n_hidden = 16

model = scglue.models.SCCLUEModel(
    adatas={"rna": adata, "metabolite": adata_desi},
    latent_dim=n_latent,
    x2u_h_dim=n_hidden,
    u2x_h_dim=n_hidden,
    random_seed=42
)

model.compile()
model.fit(adatas={"rna": adata, "metabolite": adata_desi})

In [ ]:
# DEBUG: why `n_latent` doubled?? multi-modal concatenation within `model.encode_data`? 
adata.obsm['X_glue'] = model.encode_data("rna", adata)
adata_desi.obsm['X_glue'] = model.encode_data("metabolite", adata_desi)
np.save('../results/clue_{}.npy'.format(n_latent), adata.obsm['X_glue'])

Trajectory inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/clue_{}.npy'.format(n_latent))
adata.obsm['X_glue'] = latent

In [ ]:
# Debug: why `n_latent` doubled?: Confirm w/ author: concatenation of q(z1 | x1) || q(z1 | x2)
n_nodes = 8

trajectory.get_curve(
    adata, 
    use_rep='X_glue',
    ndim=n_latent*2,
    dist_metric='euclidean', 
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_clue = adata.obs['t']
zone_clue = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_glue')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_nodes))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Spatial Gradients\n (CLUE)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (CLUE)'
)

Antibody validation:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_clue = np.array([
    compute_auroc(t_clue, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_clue = adata.uns['moranI'].loc['t', 'I']
geary_clue = adata.uns['gearyC'].loc['t', 'C']
centrality_clue = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### MOFA+

In [ ]:
import muon as mu

In [ ]:
adata.var['highly_variable'] = True
adata_desi.var['highly_variable'] = True

mdata = mu.MuData({"rna": adata, "metabolite": adata_desi})
mdata.var['highly_variable']=mdata.var['highly_variable'].to_list()
mdata

In [ ]:
n_latent = 8
mu.tl.mofa(mdata, n_factors=n_latent, seed=42)

In [ ]:
np.save('../results/mofa_{}.npy'.format(n_latent), mdata.obsm['X_mofa'])

Trajectory Inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/mofa_{}.npy'.format(n_latent))
adata.obsm['X_mofa'] = latent

In [ ]:
curve = trajectory.get_curve(adata, use_rep='X_mofa'))
trajectory.compute_pseudotime(adata, curve, root_marker='DPT')

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_mofa = adata.obs['t']
zone_mofa = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_mofa')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .03),
    vmax=np.quantile(adata.obs['t'], .97),
    cmap='RdBu_r', size=20, img=False,
    colorbar=None,
    title='Spatial Gradients\n (MOFA+)',
    # save='../results/plot/mofa.pdf'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (MOFA+)'
)

In [ ]:
# AUROC score against thresholded antibody channel
auroc_mofa = np.array([
    compute_auroc(t_mofa, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_mofa = adata.uns['moranI'].loc['t', 'I']
geary_mofa = adata.uns['gearyC'].loc['t', 'C']
centrality_mofa = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### Metrics summary

In [ ]:
# Load annotated GT
adata_ab.obs['t'] = np.load('../results/liver/antibody_gamma.npy')

# Load LYNX
adata_xenium.obsm['X_lynx'] = np.load('../results/liver/LYNX_xenium_6.npy')
curve = trajectory.get_curve(adata_xenium, use_rep='X_lynx')
trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')
adata_xenium.obs['t_lynx'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop(['t', 'seg', 'edge', 'milestones'], inplace=True, axis=1)

**Individual visualizations**

In [ ]:
# ax = sq.pl.spatial_scatter(
#     adata_xenium, color='t_lynx', size=20,
#     cmap='RdBu_r', img=False, colorbar=False, return_ax=True,
#     title=r'Inferred spatial gradient $(t)$ - LYNX'
# )
# sm = ax.collections[0]  # first scatter plot
# cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
# plt.show()

ax = sq.pl.spatial_scatter(
    adata_xenium, color='t_spaceflow', size=20, 
    cmap='RdBu_r', img=False, colorbar=False, return_ax=True,
    title=r'Inferred spatial gradient $(t)$ - SpaceFlow'
)
sm = ax.collections[0] 
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
plt.show()

ax = sq.pl.spatial_scatter(
    adata_xenium, color='t_simvi', size=20,
    cmap='RdBu_r', img=False, colorbar=False, return_ax=True,
    title=r'Inferred spatial gradient $(t)$ - SIMVI'
)
sm = ax.collections[0]  
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
plt.show()

ax = sq.pl.spatial_scatter(
    adata_xenium, color='t_gaston', size=20, 
    vmin=np.quantile(adata_xenium.obs['t_gaston'].values, .25),
    vmax=np.quantile(adata_xenium.obs['t_gaston'].values, .75),
    cmap='RdBu_r', img=False, colorbar=False, return_ax=True,
    title=r'Inferred spatial gradient $(t)$ - GASTON'
)
sm = ax.collections[0]  # first scatter plot
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
plt.show()

ax = sq.pl.spatial_scatter(
    adata_xenium, color='t_totalvi', size=20, 
    cmap='RdBu_r', img=False, colorbar=False, return_ax=True,
    title=r'Inferred spatial gradient $(t)$ - TotalVI'
)
sm = ax.collections[0]  # first scatter plot
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
plt.show()

**Continuous trajectory vs. antibody annotation**

In [ ]:
# Benchmark among ablations
plot.disp_kde_scatter(
    adata_xenium.obs['t_lynx'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"LYNX prediction $(t)$",
    title="Spatial gradient\n LYNX vs. Ground-truth"
)

# Recent ML / DL based methods
plot.disp_kde_scatter(
    adata_xenium.obs['t_simvi'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"SIMVI prediction $(t)$",
    title="Spatial gradient\n SIMVI vs. Ground-truth"
)


plot.disp_kde_scatter(
    adata_xenium.obs['t_spaceflow'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"SpaceFlow prediction $(t)$",
    title="Spatial gradient\n SpaceFlow vs. Ground-truth"
)

# plot.disp_kde_scatter(
#     adata_xenium.obs['t_coral'].values, adata_ab.obs['t'].values,
#     xlabel=r"Antibody-annotated $(t)$",
#     ylabel=r"CORAL prediction $(t)$",
#     title="Spatial gradient\n CORAL vs. Ground-truth"
# )

plot.disp_kde_scatter(
    adata_xenium.obs['t_gaston'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"GASTON prediction $(t)$",
    title="Spatial gradient\n GASTON vs. Ground-truth"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_scvi'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"scVI prediction $(t)$",
    title="Spatial gradient\n scVI vs. Ground-truth"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_totalvi'].values, adata_ab.obs['t'].values,
    xlabel=r"Antibody-annotated $(t)$",
    ylabel=r"CORAL prediction $(t)$",
    title="Spatial gradient\n TotalVI vs. Ground-truth"
)

# Basic methods
# plot.disp_kde_scatter(
#     adata_xenium.obs['t_pca'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
#     xlabel=r"Antibody-annotated $\gamma(t)$",
#     ylabel=r"PCA prediction $\gamma(t)$",
#     title="Spatial Gradient\n PCA vs. Ground-truth"
# )

# plot.disp_kde_scatter(
#     adata_xenium.obs['t_diffmap'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
#     xlabel=r"Antibody-annotated $\gamma(t)$",
#     ylabel=r"Diffmap prediction $\gamma(t)$",
#     title="Spatial Gradient\nDiffmap vs. Ground-truth"
# )

# plot.disp_kde_scatter(
#     adata_xenium.obs['t_dpt'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
#     xlabel=r"Antibody-annotated $\gamma(t)$",
#     ylabel=r"DPT prediction $\gamma(t)$",
#     title="Spatial Gradient \nDPT vs. Ground-truth"
# )


**Metrics**
- $R^2$
- AP: vs all antibody channels
- Moran's I

In [ ]:
methods = ['PCA', 'scVI', 'TotalVI', 'GASTON', 'SIMVI', 'LYNX']
ts = ['t_pca', 't_scvi', 't_totalvi', 't_gaston', 't_simvi', 't_lynx']

custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    # "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
    "#d400ff",   # Bright Purple 
]

In [ ]:
# Spatial EMD??

from sklearn.metrics import mean_squared_error
from typing import List, Iterable

def compute_rmse(
    adata : sc.AnnData,
    use_rep : Iterable[str],
    y_true : Iterable[float],
    n_repeats : int = 1,
    ss_ratio : float = 0.1
):
    rmses = np.zeros((len(use_rep), n_repeats))
    n_obs = adata.shape[0]

    for j in range(n_repeats):
        # Random subset data points
        rand_indices = np.random.choice(np.arange(n_obs), int(ss_ratio*n_obs), replace=False)
        adata_ss = adata[rand_indices]

        for i, key in enumerate(use_rep):
            y_pred = adata_ss.obs[key].values
            rmses[i, j] = mean_squared_error(y_true[rand_indices], y_pred, squared=False)
        
        del adata_ss
        gc.collect()

    return rmses
    

In [ ]:
# rmses = compute_rmse(
#     adata_xenium, 
#     y_true=adata_ab.obs['t'].values,
#     n_repeats=50,
#     use_rep=['t_pca', 't_scvi', 't_totalvi', 't_gaston', 't_simvi', 't_lynx'],
# )

# plot_df = pd.DataFrame({
#     'RMSE': rmses.flatten(),
#     'Methods': np.repeat(methods, 50),
# })

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.boxplot(plot_df, x='Methods', y='RMSE', palette=custom_palette, ax=ax)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Methods', fontsize=16)
ax.set_ylabel('RMSE', fontsize=16)
ax.set_title("Root Mean Squared Error against ground-truth gradient", fontsize=16)
ax.text(len(methods)//2-.5, 0.125, r"$lower\ is\ better$", ha='center', fontsize=10)

from statannotations.Annotator import Annotator
pairs = [('LYNX', 'SIMVI'), ('LYNX', 'GASTON')]
annotator = Annotator(ax, pairs, data=plot_df, x="Methods", y="RMSE")
annotator.configure(test='t-test_ind', text_format='star')
annotator.apply_and_annotate()

plt.show()

In [ ]:
y_gs = (adata_ab[:, ab_labels[0]].X > metrics.get_antibody_threshold(adata_ab, ab_labels[0])).squeeze().astype(np.uint8)
y_cyp = (adata_ab[:, ab_labels[1]].X > metrics.get_antibody_threshold(adata_ab, ab_labels[1])).squeeze().astype(np.uint8)
y_ass = (adata_ab[:, ab_labels[2]].X < metrics.get_antibody_threshold(adata_ab, ab_labels[2])).squeeze().astype(np.uint8)
y_col1 = (adata_ab[:, ab_labels[3]].X < metrics.get_antibody_threshold(adata_ab, ab_labels[3])).squeeze().astype(np.uint8)

y_antibodies = [y_gs, y_cyp, y_ass, y_col1]

ROC curves against `CYP` (peri-central) & `ASS1` (peri-portal) channels

In [ ]:
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_cyp, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(0.75, 0.2), loc='lower left', borderaxespad=0, fontsize=10)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against '+r'$CYP3A4$ (peri-central)', fontsize=20)
fig.show()


fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_ass, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(0.75, 0.2), loc='lower left', borderaxespad=0, fontsize=10)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against '+r'$ASS1$ (peri-portal)', fontsize=16)
fig.show()


AP scores (against all channels)

In [ ]:
ap_lynx = metrics.compute_ap(adata_xenium.obs['t_lynx'].values, y_antibodies)
ap_pca = metrics.compute_ap(adata_xenium.obs['t_pca'].values, y_antibodies)
ap_scvi = metrics.compute_ap(adata_xenium.obs['t_scvi'].values, y_antibodies)
ap_totalvi = metrics.compute_ap(adata_xenium.obs['t_totalvi'].values, y_antibodies)
ap_simvi = metrics.compute_ap(adata_xenium.obs['t_simvi'].values, y_antibodies)
# ap_coral = metrics.compute_ap(adata_xenium.obs['t_coral'].values, y_antibodies)
ap_gaston = metrics.compute_ap(adata_xenium.obs['t_gaston'].values, y_antibodies)

# aps = np.array([ap_lynx, ap_pca, ap_scvi, ap_totalvi, ap_simvi, ap_coral, ap_gaston])
aps = np.array([ap_pca, ap_scvi, ap_totalvi, ap_gaston, ap_simvi, ap_lynx])


In [ ]:
n_antibodies = ap_lynx.shape[0]
plot_df = pd.DataFrame({
    'AP': aps.flatten(),
    'Methods': np.repeat(methods, n_antibodies),
})

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.boxplot(plot_df, x='Methods', y='AP', palette=custom_palette, ax=ax)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Methods', fontsize=16)
ax.set_ylabel('AP', fontsize=16)
ax.set_title('AP score against individual antibody channels', fontsize=16)
ax.text(len(methods)//2-.5, 0.2, r"$higher\ is\ better$", ha='center', fontsize=10)

# from statannotations.Annotator import Annotator
# pairs = [('LYNX', 'SIMVI'), ('LYNX', 'GASTON')]
# annotator = Annotator(ax, pairs, data=plot_df, x="Methods", y="AP")
# annotator.configure(test='t-test_ind', text_format='full', loc='inside')
# annotator.apply_and_annotate()

plt.show()

In [ ]:
fig.savefig('../results/plots/ap.pdf', bbox_inches = "tight", dpi=300)

In [ ]:
# Moran's I
morans = metrics.compute_moran_I(
    adata_xenium, 
    n_repeats=50,
    use_rep=['t_pca', 't_scvi', 't_totalvi', 't_gaston', 't_simvi', 't_lynx'],
)

custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    # "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
    "#d400ff",  # Bright Purple 
]

plot_df = pd.DataFrame({
    'moran': morans.flatten(),
    'Methods': np.repeat(methods, morans.shape[-1])
})

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.barplot(
    data=plot_df,
    x='Methods', y='moran', 
    errorbar='sd',
    capsize=.5,
    err_kws={'linewidth': .7},
    palette=custom_palette, ax=ax
)

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Methods', fontsize=16)
ax.set_ylabel(r'I', fontsize=16)
ax.set_title("Moran's I: gradient smoothness", fontsize=16)
ax.text(len(methods)//2-.5, -0.15, r"$higher\ is\ better$", ha='center', fontsize=10)
plt.show()


In [ ]:
# Centrality
n_nodes = len(centrality_scvi)
plot_df = pd.DataFrame({
    'Degree': list(centrality_scvi) + list(centrality_multivi) + list(centrality_clue) + \
              list(centrality_mofa) + list(centrality_lynx),
    'Method': ['scVI']*n_nodes + ['MultiVI']*n_nodes + ['CLUE']*n_nodes + \
              ['MOFA+']*n_nodes + ['LYNX']*n_nodes
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.boxplot(data=plot_df, x='Method', y='Degree', hue='Method', 
                 #errorbar=("ci", 95), 
                 #err_kws={"color": ".25", "linewidth": 1},
                 ax=ax)

ax.set_title('Degree of Centrality', fontsize=15)
ax.text(2, -.16, r"$lower\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del n_nodes, plot_df

In [ ]:
disentangles = []
for i in range(len(kde_distances)):
    disentangles.append(np.sum(kde_distances[i]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

labels = methods
counts = disentangles
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

ax.bar(labels, counts, label=labels, color=colors)
ax.set_xlabel('Method')
ax.set_ylabel('Total Wasserstein distance')
ax.set_title('Disentanglement Measure:\n (Total antibody-marker distances \n along the spatial gradients)', fontsize=12)
ax.text(2, -4.5, r"$higher\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del labels, counts, colors

---